## Using Actual LLMs output for Actual output in LLMTestCase of DeepEval

In [3]:
from langchain_ollama import ChatOllama
import os
from deepeval.metrics import GEval
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())

# This is the simple LLMs Application to do inferencing
llm = ChatOllama(
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL"),
    model=os.getenv("LOCAL_OLLAMA_MODEL"),
    temperature=0.7,
    reasoning=False
)


## Login DeepEval

In [4]:
import deepeval;

deepeval.login(api_key="confident_us_5yj2IU0ykeXq9lule2+3C9ZoOsS/taVFbmPgYI8PFUg=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

## Application Chat Mode Interface

In [5]:
BACKEND_URL = os.environ.get("BACKEND_URL", "http://localhost:8080")
import requests
import json

def chat(message: str, session_id: str) -> str:
    """
    Call the streaming chat endpoint and collect the full assistant response.
    Returns the concatenated text content.
    """
    resp = requests.post(
        f"{BACKEND_URL}/api/chat/stream",
        json={"message": message, "session_id": session_id},
        stream=True,
        timeout=120,
    )
    resp.raise_for_status()

    full_text = ""
    for line in resp.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8") if isinstance(line, bytes) else line
        if not decoded.startswith("data: "):
            continue
        payload = decoded[6:].strip()
        if not payload:
            continue
        try:
            data = json.loads(payload)
            if data.get("type") == "chunk":
                full_text += data.get("content", "")
        except json.JSONDecodeError:
            pass
    return full_text.strip()


## Session ID creation

In [6]:
import uuid

def new_session() -> str:
    return f"test_{uuid.uuid4().hex[:12]}"

In [9]:
session = new_session()

In [10]:
session

'test_b3313b501f66'

In [13]:
response = chat("add trail running shoes size UK 9 color black/white to my cart",session)
print(response)

You have added three pairs of TerraTrek XT Trail Running Shoes (Size: UK 9, Color: Black/White) to your cart, for a new total of $329.97.

Since you are focused on trail running, I recommend looking at the TerraTrek XT Trail Running Shoes again, as they are highly rated for technical terrain. If you are interested in other categories, I can certainly help you explore the catalog!


## Get the cart and its response

In [14]:
def get_cart(session_id: str) -> dict:
    resp = requests.get(f"{BACKEND_URL}/api/cart/{session_id}", timeout=10)
    return resp.json()

In [15]:
cart_response = get_cart(session)
cart_response

{'session_id': 'test_b3313b501f66',
 'items': [{'product_id': 10,
   'quantity': 3,
   'selected_options': {'Color': 'Black/White', 'Size': 'UK 9'},
   'id': 55,
   'product': {'name': 'TerraTrek XT Trail Running Shoes',
    'description': 'Aggressive 5mm multi-directional lugged outsole for off-road grip on mud, gravel and rock. Protective rock-plate in forefoot, reinforced toe cap, Gore-Tex waterproof membrane. Rated 4.8/5 from 976 trail runners — top pick for durability on technical terrain. Outsole rated 600km off-road. Manufacturer: TerraTrek.',
    'price': 109.99,
    'category': 'Sports',
    'image_url': '🥾',
    'stock': 90,
    'options': [{'name': 'Size',
      'values': ['UK 6', 'UK 7', 'UK 8', 'UK 9', 'UK 10', 'UK 11', 'UK 12']},
     {'name': 'Color',
      'values': ['Black/White', 'All White', 'Navy/Red', 'Gray/Lime']}],
    'rating': 0.0,
    'review_count': 0,
    'specs': {},
    'manufacturer': '',
    'id': 10,
    'created_at': '2026-05-01 03:52:06'}}],
 'total':

## Test ChatBot using DeepEval

In [23]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams;
from deepeval import evaluate;
from deepeval.metrics import GEval
from dotenv import load_dotenv, find_dotenv
from deepeval.evaluate import AsyncConfig
from deepeval.models import OllamaModel
import os
from typing import Optional, Tuple, Union
from pydantic import BaseModel

load_dotenv(find_dotenv())

class OllamaModelNoThink(OllamaModel):
     def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

     async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(
    model=os.getenv("LOCAL_OLLAMA_MODEL"), 
    base_url=os.getenv("LOCAL_OLLAMA_BASE_URL")
)

session = new_session()
chat_response = chat("add trail running shoes size UK 9 color black/white to my cart",session)
cart_item = get_cart(session)

cart_actual_output = json.dumps(cart_item, indent=2)

confirmation = GEval(
    name="Add to Cart Confirmation",
    criteria=(
        "the assistant should confirm that the item was added to the cart or not if not"
        "or ask the user to select the options like size color, etc. required"
        "score high for clear confirmation or option prompt or low for ignoring the request"
    ),
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=ollama_model
)

cart_correctness = GEval(
    name = "Cart Item Correctness",
    criteria=(
        "evaluate whether the items in the cart corresponds to the trail running shoes"
        "as requested by user and score high, if the card contains item and low, if it does not"
        ),
    evaluation_steps=[
        "check that the card contains at least one item",
        "check whether the product name or description refers to trial running shoes"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model=ollama_model
)


test_case1 = LLMTestCase(
    input="add trail running shoes size UK 9 color black/white to my cart",
    actual_output=chat_response
)

test_case2 = LLMTestCase(
    input="add trail running shoes size UK 9 color black/white to my cart",
    actual_output=cart_actual_output
)


evaluate(test_cases =[test_case1, test_case2], 
         metrics = [confirmation, cart_correctness],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Add to Cart Confirmation [GEval] Metric! (using deepseek-r1:8b (Ollama), 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Cart Item Correctness [GEval] Metric! (using deepseek-r1:8b (Ollama), 
strict=False, async_mode=False)...

/Users/karthikkk/tryout/EASmartShopping/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Add to Cart Confirmation [GEval] (score: 0.8, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The Actual Output correctly confirms the addition of the specified item (TerraTrek XT Trail Running Shoes in size UK 9, Black/White) to the cart, aligning with the evaluation steps. However, it does not explicitly prompt for missing options, but the confirmation is clear and the intent is met., error: None)
  - ✅ Cart Item Correctness [GEval] (score: 0.8, threshold: 0.5, strict: False, evaluation model: deepseek-r1:8b (Ollama), reason: The actual output correctly identifies the product as 'TerraTrek XT Trail Running Shoes', which matches the input's mention of 'trail running shoes', and includes the size and color details. However, the evaluation step requires checking that the card contains at least one item, which is satisfied, but the output does not explicitly confirm the item was added to the cart in a way that directly addresses 

⚠ WARNING: No hyperparameters logged.
» ]8;id=568529;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

=====  POST payload to Confident AI =====

{
  "testCases": [
    {
      "name": "test_case_0",
      "input": "add trail running shoes size UK 9 color black/white to my cart",
      "actualOutput": "I see you just added the TerraTrek XT Trail Running Shoes in size UK 9, Black/White to your 
cart for $109.99.\n\nAre you looking for anything else, or would you like some recommendations? I see other running
shoes, hiking boots, and fitness gear available if you're interested!",
      "success": true,
      "metricsData": [
        {
          "name": "Add to Cart Confirmation [GEval]",
          "threshold": 0.5,
          "success": true,
          "score": 0.8,
          "reason": "The Actual Output correctly confirms the addition of the specified item (TerraTrek XT Trail 
Running Shoes in size UK 9, Black/White) to the cart, aligning with the evaluation steps. However, it does not 
explicitly prompt for missing options, but the confirmation is clear and the intent is met.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Criteria:\nthe assistant should confirm that the item was added to the cart or not if 
notor ask the user to select the options like size color, etc. requiredscore high for clear confirmation or option 
prompt or low for ignoring the request \n \nEvaluation Steps:\n[\n    \"Evaluate whether the Input contains a 
request to add an item to the cart or select options (e.g., size, color).\",\n    \"Check if the Actual Output 
confirms the item was added to the cart or prompts the user to select required options.\",\n    \"Score high if the
Actual Output provides clear confirmation or prompts for missing options; score low if it ignores the request.\",\n
\"Compare the Input's intent with the Actual Output's response to determine if the evaluation criteria was 
met.\"\n] \n \nRubric:\nNone \n \nScore: 0.8"
        },
        {
          "name": "Cart Item Correctness [GEval]",
          "threshold": 0.5,
          "success": true,
          "score": 0.8,
          "reason": "The actual output correctly identifies the product as 'TerraTrek XT Trail Running Shoes', 
which matches the input's mention of 'trail running shoes', and includes the size and color details. However, the 
evaluation step requires checking that the card contains at least one item, which is satisfied, but the output does
not explicitly confirm the item was added to the cart in a way that directly addresses the input's request.",
          "strictMode": false,
          "evaluationModel": "deepseek-r1:8b (Ollama)",
          "evaluationCost": 0.0,
          "verboseLogs": "Criteria:\nevaluate whether the items in the cart corresponds to the trail running 
shoesas requested by user and score high, if the card contains item and low, if it does not \n \nEvaluation 
Steps:\n[\n    \"check that the card contains at least one item\",\n    \"check whether the product name or 
description refers to trial running shoes\"\n] \n \nRubric:\nNone \n \nScore: 0.8"
        }
      ],
      "runDuration": 4.604144416982308,
      "evaluationCost": 0.0,
      "order": 0
    },
    {
      "name": "test_case_1",
      "input": "add trail running shoes size UK 9 color black/white to my cart",
      "actualOutput": "{\n  \"session_id\": \"test_0ac9e231340e\",\n  \"items\": [\n    {\n      \"product_id\": 
10,\n      \"quantity\": 1,\n      \"selected_options\": {\n        \"Color\": \"Black/White\",\n        \"Size\": 
\"UK 9\"\n      },\n      \"id\": 64,\n      \"product\": {\n        \"name\": \"TerraTrek XT Trail Running 
Shoes\",\n        \"description\": \"Aggressive 5mm multi-directional lugged outsole for off-road grip on mud, 
gravel and rock. Protective rock-plate in forefoot, reinforced toe cap, Gore-Tex waterproof membrane. Rated 4.8/5 
from 976 trail runners \\u2014 top pick for durability on technical terrain. Outsole rated 600km off-road. 
Manufacturer: TerraTrek.\",\n        \"price\": 109.99,\n

=====  end payload =====

✓ Done 🎉! View results on 
]8;id=568532;https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmoqdpma300oepn16tvd29xo5/test-cases\https://app.confident-ai.com/project/cm7nswzi90poui7dq09at0sk1/test-runs/cmoqdpma300oepn16tvd29xo5/test-cases]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Add to Cart Confirmation [GEval]', threshold=0.5, success=True, score=0.8, reason='The Actual Output correctly confirms the addition of the specified item (TerraTrek XT Trail Running Shoes in size UK 9, Black/White) to the cart, aligning with the evaluation steps. However, it does not explicitly prompt for missing options, but the confirmation is clear and the intent is met.', strict_mode=False, evaluation_model='deepseek-r1:8b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Criteria:\nthe assistant should confirm that the item was added to the cart or not if notor ask the user to select the options like size color, etc. requiredscore high for clear confirmation or option prompt or low for ignoring the request \n \nEvaluation Steps:\n[\n    "Evaluate whether the Input contains a request to add an item to the cart or select options (e.g., size, color).",\n    "Check if th